In [1]:
import os
import pickle
import numpy as np
import pandas as pd

import scanpy as sc
import mudata

from scenicplus.data_wrangling.adata_cistopic_wrangling import process_multiome_data

/opt/conda/envs/scenicplus/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-14 05:03:27,655	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
data_dir = "/scenic_plus_vol/SCENIC/full_dataset"
cisTopic_obj_fname = f"{data_dir}/cistopic_obj.pkl"
GEX_anndata_fname = f"{data_dir}/GEX.h5ad"

In [ ]:
GEX_anndata = mudata.read(GEX_anndata_fname.__str__())

with open(cisTopic_obj_fname, "rb") as f:
    cisTopic_obj = pickle.load(f)

In [ ]:
all_cells = GEX_anndata.obs.index.values
shuffled_cells = np.random.permutation(all_cells)

# Split cells into 4 (approximately) equally sized groups
cells_per_replicate = np.array_split(shuffled_cells, 4)
for rep in range(len(cells_per_replicate)):
    out_dir = f"{data_dir}/downsampled_rep{rep+1}"
    os.mkdir(out_dir)
    pd.Series(cells_per_replicate[rep]).to_csv(f"{out_dir}/cells.tsv", index=False, header=False)

In [ ]:
bc_transform_func = lambda x: f'{x}___cisTopic'

for rep in range(len(cells_per_replicate)):
    print(rep)
    out_dir = f"{data_dir}/downsampled_rep{rep+1}"
    out_file = f"{out_dir}/ACC_GEX.h5mu"

    # manually process rna+atac into combined object
    # although this step is also part of the snakemake pipeline,
    # this saves some storage space by not separately saving the rna and atac subsets
    cells = pd.read_csv(f"{out_dir}/cells.tsv", header=None)[0].values
    GEX_anndata_subset = GEX_anndata[cells,:]
    mdata = process_multiome_data(
        GEX_anndata=GEX_anndata_subset,
        cisTopic_obj=cisTopic_obj,
        use_raw_for_GEX_anndata=True,
        bc_transform_func=bc_transform_func)
    mdata.write_h5mu(out_file)

    del mdata
    del GEX_anndata_subset

2
2025-08-14 02:53:21,219 Ingesting multiome data INFO     Found 215851 multiome cells.
2025-08-14 02:53:21,992 cisTopic     INFO     Imputing region accessibility
2025-08-14 02:53:21,993 cisTopic     INFO     Impute region accessibility for regions 0-20000
2025-08-14 02:53:30,041 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2025-08-14 02:53:38,090 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2025-08-14 02:53:46,129 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2025-08-14 02:53:54,132 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2025-08-14 02:54:02,157 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2025-08-14 02:54:10,191 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2025-08-14 02:54:18,215 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2025-08-14 02:54:26,257 cisTopic     INFO     Imp

... storing 'sample_id' as categorical
... storing 'Chromosome' as categorical


3
2025-08-14 03:40:07,764 Ingesting multiome data INFO     Found 215850 multiome cells.
2025-08-14 03:40:08,385 cisTopic     INFO     Imputing region accessibility
2025-08-14 03:40:08,386 cisTopic     INFO     Impute region accessibility for regions 0-20000
2025-08-14 03:40:16,397 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2025-08-14 03:40:24,388 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2025-08-14 03:40:32,374 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2025-08-14 03:40:40,329 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2025-08-14 03:40:48,308 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2025-08-14 03:40:56,301 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2025-08-14 03:41:04,290 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2025-08-14 03:41:12,299 cisTopic     INFO     Imp

... storing 'sample_id' as categorical
... storing 'Chromosome' as categorical


### Manually run AUCell_direct

In [5]:
import os
from typing import Literal
from tqdm import tqdm

import numpy as np
import pandas as pd

import mudata

from scenicplus.eregulon_enrichment import get_eRegulons_as_signatures, rank_data, ranked_data
from pycisTopic.signature_enrichment import signature_enrichment

# reimplentation of rank_data from scenicplus.eregulon_enrichment that is more memory efficient
def rank_data_new(
        mtx: np.ndarray,
        index: np.ndarray,
        columns: np.ndarray,
        axis: Literal[0, 1] = 1,
        seed: int = 123) -> ranked_data:
    """
    """
    # Initialize random number generator, for handling ties
    rng = np.random.default_rng(seed=seed)

    # Function to make rankings per array
    def rank_scores_and_assign_random_ranking_in_range_for_ties(
            scores_with_ties_for_motif_or_track_numpy: np.ndarray
    ) -> np.ndarray:
        #
        # Create random permutation so tied scores will have a different ranking each time.
        random_permutations_to_break_ties_numpy = rng.permutation(
            scores_with_ties_for_motif_or_track_numpy.shape[0]
        )
        ranking_with_broken_ties_for_motif_or_track_numpy = random_permutations_to_break_ties_numpy[
            (-scores_with_ties_for_motif_or_track_numpy)[
                random_permutations_to_break_ties_numpy].argsort()
        ].argsort().astype(imputed_acc_obj_ranking_db_dtype)

        return ranking_with_broken_ties_for_motif_or_track_numpy

    # Get dtype of the scores
    imputed_acc_obj_ranking_db_dtype = 'uint32'
    # mtx = df.to_numpy()  # save memory by not converting to numpy
    ranking = np.zeros_like(mtx, dtype=imputed_acc_obj_ranking_db_dtype)

    if axis == 0:
        for i in tqdm(range(mtx.shape[1])):
            ranking[:, i] = rank_scores_and_assign_random_ranking_in_range_for_ties(
                mtx[:, i])
    elif axis == 1:
        for i in tqdm(range(mtx.shape[0])):
            ranking[i, :] = rank_scores_and_assign_random_ranking_in_range_for_ties(
                mtx[i, :])
    else:
        raise ValueError(f"Axis can only be 0 or 1 not {axis}")

    return ranked_data(
        mtx=ranking.T,
        feature_names=columns if axis == 1 else index,
        cell_names=index if axis == 1 else columns)

In [ ]:
data_dir = "/scenic_plus_vol/SCENIC/full_dataset"
auc_threshold = 0.05
normalize = False
n_cpu = 24

for rep in [4]:
    print(f"Processing rep {rep}")
    multiome_mudata_fname = f"{data_dir}/downsampled_rep{rep}/ACC_GEX.h5mu"
    eRegulons_fname = f"{data_dir}/downsampled_rep{rep}/eRegulon_direct.tsv"
    out_file = f"{data_dir}/downsampled_rep{rep}/AUCell_direct.h5mu"
    out_file_gene = f"{data_dir}/downsampled_rep{rep}/AUCell_direct_gene.h5mu"
    out_file_region = f"{data_dir}/downsampled_rep{rep}/AUCell_direct_region.h5mu"

    print("Reading data")
    mdata = mudata.read(multiome_mudata_fname.__str__())
    eRegulons = pd.read_table(eRegulons_fname)
    eRegulon_signatures = get_eRegulons_as_signatures(eRegulons=eRegulons)
    del eRegulons

    print("Ranking gex data")
    gex_mtx = mdata["scRNA"].to_df()
    gex_ranking = rank_data(gex_mtx)
    del gex_mtx

    print("Calculating gex AUC")
    gex_AUC = signature_enrichment(
        gex_ranking,
        eRegulon_signatures["Gene_based"],
        enrichment_type='gene',
        auc_threshold=auc_threshold,
        normalize=normalize,
        n_cpu=n_cpu)
    del gex_ranking
    mdata_AUC_gene = mudata.MuData(
        {
            "Gene_based": mudata.AnnData(X=gex_AUC),
        })
    del gex_AUC
    mdata_AUC_gene.write_h5mu(out_file_gene.__str__())
    del mdata_AUC_gene
    
    print("Ranking acc data")
    acc_mtx = mdata["scATAC"].X
    acc_index = mdata["scATAC"].obs.index.values
    acc_columns = mdata["scATAC"].var.index.values
    del mdata
    acc_ranking = rank_data_new(acc_mtx, acc_index, acc_columns)
    del acc_mtx, acc_index, acc_columns

    print("Calculating acc AUC")
    acc_AUC = signature_enrichment(
        acc_ranking,
        eRegulon_signatures["Region_based"],
        enrichment_type='gene',
        auc_threshold=auc_threshold,
        normalize=normalize,
        n_cpu=n_cpu)
    del acc_ranking
    del eRegulon_signatures

    print("Write mudata object")
    mdata_AUC_region = mudata.MuData(
        {
            "Region_based": mudata.AnnData(X=acc_AUC)
        })
    del acc_AUC
    mdata_AUC_region.write_h5mu(out_file_region.__str__())
    del mdata_AUC_region
    

Processing rep 4
Reading data


/opt/conda/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/opt/conda/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Ranking acc data


100%|██████████| 215850/215850 [1:37:54<00:00, 36.74it/s]


Calculating acc AUC
Write mudata object


In [6]:
data_dir = "/scenic_plus_vol/SCENIC/full_dataset"

for rep in range(1, 5):
    print(f"Processing rep {rep}")
    out_file = f"{data_dir}/downsampled_rep{rep}/AUCell_direct.h5mu"
    out_file_gene = f"{data_dir}/downsampled_rep{rep}/AUCell_direct_gene.h5mu"
    out_file_region = f"{data_dir}/downsampled_rep{rep}/AUCell_direct_region.h5mu"
    if not os.path.exists(out_file):
        gene_data = mudata.read(out_file_gene)
        region_data = mudata.read(out_file_region)

        mdata_AUC = mudata.MuData(
            {
                "Gene_based": gene_data["Gene_based"],
                "Region_based": region_data["Region_based"]
            })
        mdata_AUC.write_h5mu(out_file)
            

Processing rep 1


/opt/conda/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/opt/conda/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Processing rep 2
Processing rep 3


/opt/conda/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/opt/conda/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Processing rep 4


/opt/conda/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/opt/conda/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
